# S15: Reducción de dimensionalidad y balanceo

Pipeline para `novamarket_features_seleccionadas.csv`: separación estratificada, imputación y escalado sin fuga de datos, diagnóstico de correlación, PCA condicional, comparación de técnicas de balanceo y exportación de los conjuntos finales.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

DATASET_FILENAME = "novamarket_features_seleccionadas.csv"
starting_dir = Path.cwd().resolve()
project_root = next(
    (path for path in (starting_dir, *starting_dir.parents)
     if (path / "S14" / DATASET_FILENAME).exists() or (path / DATASET_FILENAME).exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        f"No se encontró {DATASET_FILENAME} desde {starting_dir} ni en sus carpetas padre."
    )

ruta_csv = project_root / "S14" / DATASET_FILENAME
if not ruta_csv.exists():
    ruta_csv = project_root / DATASET_FILENAME
output_dir = project_root / "S15"
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ruta_csv)
columnas_requeridas = {"fuga_cliente", "id_cliente"}
faltantes = columnas_requeridas - set(df.columns)
if faltantes:
    raise ValueError(f"Faltan columnas requeridas: {sorted(faltantes)}")

X = df.drop(columns=["fuga_cliente", "id_cliente"])
y = df["fuga_cliente"]
if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in X.dtypes):
    raise TypeError("Las features deben ser numéricas antes de imputar, escalar y aplicar PCA/SMOTE.")
if not set(y.dropna().unique()).issubset({0, 1}):
    raise ValueError("La columna fuga_cliente debe contener etiquetas binarias 0/1.")
if y.isna().any():
    raise ValueError("La columna objetivo fuga_cliente contiene valores faltantes.")

print(f"Dataset: {ruta_csv}")
print(f"Filas: {len(df):,} | Features predictoras: {X.shape[1]}")
display(df.head())

## 1. Separación estratificada

Se separa el 20% para prueba antes de ajustar transformaciones o aplicar balanceo. La estratificación mantiene la distribución de `fuga_cliente` en ambos subconjuntos.

**Comprobación de los datos:** el archivo de entrada tiene una prevalencia de fuga de aproximadamente **20,31%**, no cercana al 9%. Se conserva la proporción real del dataset sin cambiar las etiquetas; las proporciones impresas por la celda siguiente permiten comprobarlo.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=df["fuga_cliente"],
)

print("Proporción de fuga_cliente en entrenamiento:")
display(y_train.value_counts(normalize=True).sort_index().rename("proporción"))
print("Proporción de fuga_cliente en prueba:")
display(y_test.value_counts(normalize=True).sort_index().rename("proporción"))
print("Conteos de entrenamiento:")
display(y_train.value_counts().sort_index().rename("filas"))

## 2. Imputación y escalado sin fuga de datos

El dataset contiene valores faltantes y también valores infinitos en las features. Estos últimos se convierten en faltantes y se imputan con la mediana. Tanto el imputador como `StandardScaler` se ajustan solo con `X_train`; esos mismos ajustes se reutilizan para transformar `X_test`. Así, la prueba no aporta información al entrenamiento del preprocesamiento.

In [ ]:
inf_train = np.isinf(X_train.to_numpy(dtype=float))
inf_test = np.isinf(X_test.to_numpy(dtype=float))
valores_infinitos = int(inf_train.sum() + inf_test.sum())
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
print(f"Valores infinitos tratados como faltantes: {valores_infinitos}")

imputer = SimpleImputer(strategy="median")
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test), columns=X_test.columns, index=X_test.index
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_imputed),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_imputed),
    columns=X_test.columns,
    index=X_test.index,
)

assert X_train_scaled.select_dtypes(exclude=np.number).empty
assert X_test_scaled.select_dtypes(exclude=np.number).empty
print("Features numéricas verificadas; imputador y escalador ajustados solo con entrenamiento.")

## 3. Correlación y decisión de PCA

La correlación se calcula sobre las features originales de `X_train` (`X_train.corr()`). Se consideran redundantes los pares cuya correlación absoluta supera 0.6. Si se detectan, PCA combina variables correlacionadas en componentes y conserva el 90% de la varianza; si no, se omite para preservar la representación original.

El escalador se ajusta solo en entrenamiento antes del PCA. PCA también se ajusta exclusivamente con entrenamiento; la prueba se transforma usando esos mismos ajustes, sin volver a estimarlos.

In [ ]:
corr_matrix = X_train.corr()
display(corr_matrix)

corr_pairs = [
    (col1, col2, corr_matrix.loc[col1, col2])
    for i, col1 in enumerate(corr_matrix.columns)
    for col2 in corr_matrix.columns[i + 1:]
    if pd.notna(corr_matrix.loc[col1, col2]) and abs(corr_matrix.loc[col1, col2]) > 0.6
]
corr_pairs.sort(key=lambda pair: abs(pair[2]), reverse=True)
correlaciones_altas = pd.DataFrame(
    corr_pairs, columns=["feature_1", "feature_2", "correlación"]
)
print("Pares con |correlación| > 0.6:")
display(correlaciones_altas)

pca = None
if corr_pairs:
    pca = PCA(n_components=0.90)
    X_train_model = pd.DataFrame(
        pca.fit_transform(X_train_scaled),
        index=X_train_scaled.index,
        columns=[f"PC{i + 1}" for i in range(pca.n_components_)],
    )
    X_test_model = pd.DataFrame(
        pca.transform(X_test_scaled),
        index=X_test_scaled.index,
        columns=X_train_model.columns,
    )
    varianza_explicada = float(pca.explained_variance_ratio_.sum())
    print(f"PCA aplicado: {pca.n_components_} componentes; varianza explicada {varianza_explicada:.1%}.")
    display(pd.Series(pca.explained_variance_ratio_.cumsum(), name="varianza acumulada"))
else:
    X_train_model = X_train_scaled.copy()
    X_test_model = X_test_scaled.copy()
    varianza_explicada = None
    print("PCA omitido: no hay pares con |correlación| > 0.6.")

## 4. Comparación de técnicas de balanceo

Las tres alternativas se aplican solo al conjunto de entrenamiento y se comparan por tamaño y número de ejemplos por clase:

- **RandomUnderSampler:** reduce la clase mayoritaria; puede perder información útil.
- **RandomOverSampler:** replica ejemplos minoritarios; conserva la mayoría, pero puede favorecer sobreajuste por duplicación.
- **SMOTE:** sintetiza ejemplos minoritarios a partir de vecinos cercanos; evita limitarse a duplicar registros, aunque puede producir combinaciones poco realistas.

Todas usan `random_state=42`. El conjunto de prueba permanece intacto y no se balancea.

In [ ]:
if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in X_train_model.dtypes):
    raise TypeError("X_train_model debe ser completamente numérico para aplicar balanceo.")

conteo_minoria = int(y_train.value_counts().min())
if conteo_minoria < 2:
    raise ValueError("SMOTE requiere al menos dos observaciones de la clase minoritaria.")
k_vecinos = min(5, conteo_minoria - 1)

X_train_under, y_train_under = RandomUnderSampler(random_state=42).fit_resample(
    X_train_model, y_train
)
X_train_over, y_train_over = RandomOverSampler(random_state=42).fit_resample(
    X_train_model, y_train
)
X_train_smote, y_train_smote = SMOTE(
    random_state=42, k_neighbors=k_vecinos
).fit_resample(X_train_model, y_train)

def resumen_muestreo(nombre, X_res, y_res):
    conteos = y_res.value_counts().sort_index()
    return {
        "técnica": nombre,
        "filas": len(y_res),
        "clase 0": int(conteos.get(0, 0)),
        "clase 1": int(conteos.get(1, 0)),
        "features duplicadas": int(X_res.duplicated().sum()),
    }

comparacion = pd.DataFrame([
    resumen_muestreo("RandomUnderSampler", X_train_under, y_train_under),
    resumen_muestreo("RandomOverSampler", X_train_over, y_train_over),
    resumen_muestreo("SMOTE", X_train_smote, y_train_smote),
])
display(comparacion)

## 5. Elección y exportación

Se elige **SMOTE** para el entrenamiento final porque equilibra las clases sin descartar registros mayoritarios y agrega variedad a la clase minoritaria. Esta elección es una decisión de preprocesamiento, no una garantía de mejor desempeño: debe validarse con métricas apropiadas para clases desbalanceadas.

La prueba se exporta con sus columnas, valores y proporción originales para que la evaluación posterior refleje los datos reales.

In [ ]:
# El conjunto de entrenamiento exportado contiene las features finales y la etiqueta.
X_final = pd.DataFrame(X_train_smote, columns=X_train_model.columns).reset_index(drop=True)
y_final = pd.Series(y_train_smote, name="fuga_cliente").reset_index(drop=True)
train_balanceado = pd.concat([X_final, y_final], axis=1)

# Prueba intacta: registros, columnas, valores y proporción originales; sin escalado ni balanceo.
test_export = df.loc[X_test.index].sort_index().copy()
train_path = output_dir / "novamarket_train_balanceado.csv"
test_path = output_dir / "novamarket_test.csv"
train_balanceado.to_csv(train_path, index=False)
test_export.to_csv(test_path, index=False)

assert train_balanceado["fuga_cliente"].value_counts().nunique() == 1
assert test_export.equals(df.loc[test_export.index])
print(f"Entrenamiento balanceado: {train_path} | {train_balanceado.shape}")
print(f"Prueba original intacta: {test_path} | {test_export.shape}")
print("Proporciones en la prueba exportada:")
display(test_export["fuga_cliente"].value_counts(normalize=True).sort_index())

# Decisiones de reducción y balanceo

## División, datos faltantes y prevención de fuga

Se separó el 20% para prueba mediante `train_test_split` con `stratify=df["fuga_cliente"]` y `random_state=42`. Entrenamiento: clase 0 = 351.784 y clase 1 = 89.648. Prueba: clase 0 = 87.946 y clase 1 = 22.412; la prevalencia de fuga en prueba es 20,31%. El CSV de entrada tiene una prevalencia positiva de 20,31%, no el ~9% esperado; la estratificación conserva la distribución observada sin alterar las etiquetas.

El identificador `id_cliente` se excluyó de las features predictoras. Los infinitos en features se trataron como faltantes y se imputaron con la mediana, ajustada solo en entrenamiento. `SimpleImputer` y `StandardScaler` se ajustaron exclusivamente con `X_train`; los mismos ajustes transformaron la prueba. El conjunto de prueba no se usó para ajustar imputación, escalado, PCA ni técnicas de balanceo.

## Correlación y PCA

La matriz de correlación se calculó como `X_train.corr()`. Criterio: usar PCA si al menos un par tiene `|correlación| > 0.6`.

Se aplicó PCA porque se detectaron correlaciones absolutas mayores a 0.6 (`monto_compra`–`unidades_vendidas`: 0,684; `monto_compra`–`monto_por_unidad`: 0,613). Se conservaron 7 componentes, que explican 99,0% de la varianza. PCA se ajustó solo con entrenamiento.

PCA es una transformación lineal que combina features y puede reducir redundancia y colinealidad. Se usó `n_components=0.90` para conservar al menos el 90% de la varianza; el ajuste se hizo solo con entrenamiento y se aplicó sin reajustes a la prueba.

## Comparación del balanceo

Las tres técnicas se aplicaron solo a entrenamiento con `random_state=42`:

| Técnica | Filas | Clase 0 | Clase 1 | Features duplicadas |
| --- | ---: | ---: | ---: | ---: |
| RandomUnderSampler | 179.296 | 89.648 | 89.648 | 869 |
| RandomOverSampler | 703.568 | 351.784 | 351.784 | 265.244 |
| SMOTE | 703.568 | 351.784 | 351.784 | 3.702 |

- **RandomUnderSampler:** reduce la clase mayoritaria, pero puede descartar información útil.
- **RandomOverSampler:** replica casos de la minoritaria y conserva los datos originales, aunque puede aumentar el sobreajuste por duplicación.
- **SMOTE:** crea ejemplos sintéticos de la minoritaria a partir de vecinos cercanos, manteniendo la mayoría y evitando duplicados exactos. Puede crear combinaciones poco realistas; sus resultados deben validarse con métricas apropiadas para clases desbalanceadas.

Se eligió **SMOTE** para el entrenamiento final por el balance entre retener los datos mayoritarios y diversificar la clase minoritaria. La prueba se mantuvo sin balancear para evaluar posteriormente con su prevalencia real.

## Archivos

- `novamarket_train_balanceado.csv`: features finales de entrenamiento y etiqueta `fuga_cliente`; `id_cliente` queda excluido.
- `novamarket_test.csv`: filas de prueba originales, con todas sus columnas y valores, sin escalar ni balancear.

In [ ]:
import json

notebook_path = output_dir / "S15_pca_muestreo.ipynb"
notebook_data = json.loads(notebook_path.read_text(encoding="utf-8"))
contenido_md = next(
    "".join(cell["source"])
    for cell in notebook_data["cells"]
    if "decision-report" in cell.get("metadata", {}).get("tags", [])
)
decision_path = output_dir / "S15_decisiones_reduccion_balanceo.md"
decision_path.write_text(contenido_md.strip() + "\n", encoding="utf-8")
print(f"Justificaciones guardadas: {decision_path}")